# 02b｜保留试跑成果并续训至累计 300 步

130U-FinLLM｜130U 金融大语言模型

先完成 02a。本页必须恢复 seed_v1 的已有检查点，先验证第 61→62 步的断点续训，再训练至累计 300 次更新（正常更新约 4,915,200 tokens）。所有模型、数据、优化器、学习率调度和精度设置保持原样，只延长停止步数。原始 61 步试跑报告会在 Drive 独立归档；不重新初始化模型。

每个 Notebook 都包含独立环境入口。Colab 菜单：运行时 → 更改运行时类型 → T4 GPU。点击执行代码前可查看源码；所有输出均为实际执行后产生，仓库不预填训练成绩。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")

In [ ]:
CONTINUATION_SUCCEEDED = False
RESUME_PROBE_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
SEED_VERSION = "seed_v1"
PILOT_UPDATES = 61
PROBE_UPDATES = 62
TARGET_UPDATES = 300
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 运行时；不能在 CPU 上延续 T4 检查点。")
from finllm.continuation import prepare_seed_continuation, run_seed_segment, backup_seed_continuation
continuation = prepare_seed_continuation(
    PROJECT, DRIVE_ROOT, SEED_VERSION, pilot_updates=PILOT_UPDATES,
    probe_updates=PROBE_UPDATES, target_updates=TARGET_UPDATES)
print("已恢复更新步数：", continuation["restored_updates"])
print("原始试跑报告已验证并归档：", continuation["pilot_archive"])
print("GPU:", torch.cuda.get_device_name(0))

先执行一次最短续训验证。已经完成第 62 步或后续步骤时，本单元格只校验并读取现有进度，不倒退、不重跑。若 Colab 的 PyTorch/CUDA 等环境与原检查点不同，会明确停止；请保留报错，不要通过改精度、批次大小或重新初始化绕过检查。

In [ ]:
RESUME_PROBE_SUCCEEDED = False
CONTINUATION_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
probe_result = run_seed_segment(continuation, "probe", finllm)
print(json.dumps(probe_result, indent=2, ensure_ascii=False))
RESUME_PROBE_SUCCEEDED = True

续训到累计 300 步，包含原先的 61 步。原始 pilot_config.json 保持不变。每 10 次更新自动备份检查点；若中断，重新从本页第一格执行即可恢复已保存进度。各段报告中的 updates/tokens 是累计计数，elapsed_seconds/tokens_per_second 则只计该段实际新训练。

In [ ]:
CONTINUATION_SUCCEEDED = False
CONTINUATION_BACKED_UP = False
if not globals().get("RESUME_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先成功完成上方断点续训验证。")
continuation_result = run_seed_segment(continuation, "continue", finllm)
print(json.dumps(continuation_result, indent=2, ensure_ascii=False))
CONTINUATION_SUCCEEDED = True

In [ ]:
CONTINUATION_BACKED_UP = False
if not globals().get("CONTINUATION_SUCCEEDED", False):
    raise RuntimeError("续训尚未成功结束；保留现有备份，先处理上方错误。")
backup_result = backup_seed_continuation(continuation)
print(json.dumps(backup_result, indent=2, ensure_ascii=False))
CONTINUATION_BACKED_UP = True
print("累计续训与校验备份完成：", continuation["seed_drive"])
print("原始试跑报告：", continuation["pilot_archive"])

完成后保留本页的续训报告与 verified: true 备份输出，再比较验证 loss 和资源消耗来决定下一段语料及预算。本页不会自动进入领域训练、SFT 或 DPO；不要修改原始 01a/02a 配置或删除 seed_v1 备份。